# Verification notebook: World Bank SPI v4.0

Runs the package scripts and displays the main results. Pinned commit `2dd02f746cb4e2159281adc74d6865f948e90ce1`. Cells are not pre-executed; run them top to bottom from the package root.

Scope: consistency of the published pipeline, not the quality of the underlying statistics.

In [ ]:
import subprocess, sys, pandas as pd
subprocess.run([sys.executable, 'scripts/run_all.py'], check=True)   # use '--online' to download from the pinned commit instead

## 1. Stage-wise reconstruction (Table 6)

In [ ]:
s = pd.read_csv('results/e2e_field_summary.csv')
s['stage_no'] = s.stage.str.split().str[0]
s.groupby('stage_no').agg(fields=('field','count'), cells=('n_nonmissing','sum'), mismatches=('n_mismatch','sum'), max_abs_diff=('max_abs_diff','max'))

## 2. Geospatial pre-rule (165 cells explained)

In [ ]:
g = pd.read_csv('results/geo_discrepancy_165_cells.csv'); print(len(g), 'cells,', g.iso3c.nunique(), 'economies'); g.head()

## 3. Revisions across releases (Table 8)

In [ ]:
v = pd.read_csv('results/vintage_revisions_summary.csv'); v[v.field=='SPI.INDEX'][['older','newer','n_common','n_revised','share_revised','mean_abs_revised']]

## 4. Uzbekistan: overall score by release

In [ ]:
o = pd.read_csv('results/vintage_overall_panel.csv'); o[(o.iso3c=='UZB')].sort_values('year')

## 5. Ranks 2024 and rounding sensitivity

In [ ]:
r = pd.read_csv('results/rank_2024_188.csv'); print((r.rank_min!=r.rank_min_rounded2).sum(), 'rank changes at 2 decimals;', (r.rank_min!=r.rank_min_rounded1).sum(), 'at 1 decimal'); r[r.iso3c=='UZB']

## 6. Claims register (Table 9)

In [ ]:
c = pd.read_csv('results/project_claims_audit.csv'); c.groupby(['Claim_type','Status_category']).size().unstack(fill_value=0)

## Applying to another economy
The reconstruction covers all economies in the file. For a claims register on another report, create a CSV with the columns of `claims/claims_input_original.csv` and run `scripts/claims_register_postprocess.py IN OUT`.